# Tulpar üslup eğitimi (Gemma 4 26B-A4B, LoRA)

**Çalışma zamanı:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU*. Sırayla her hücreyi çalıştır (▶).

Bu defter senin maskelenmiş mesaj örneklerinle Gemma'ya **senin üslubunu** öğretir. Çıktı küçük bir LoRA dosyasıdır; HP'deki mevcut modelin üstüne takılır, modeli değiştirmez.

Süre: kurulum ~10 dk, eğitim ~2-3 saat (19 bin örnek, 1 tur). Sekmeyi açık tut.

In [ ]:
# 1) Ekran kartını kontrol et: A100 görmelisin
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Kurulum (~5 dk)
%%capture
!pip install -U unsloth

## 3) Veriyi yükle
Sol taraftaki 📁 simgesine tıklayıp `egitim.jsonl` ve `dogrulama.jsonl` dosyalarını sürükle bırak, ya da aşağıdaki hücreyi çalıştırıp seç.

In [ ]:
import os
from google.colab import files
if not (os.path.exists("egitim.jsonl") and os.path.exists("dogrulama.jsonl")):
    files.upload()
!wc -l egitim.jsonl dogrulama.jsonl

In [ ]:
# 4) Modeli yükle (4-bit, ~15 GB)
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-26B-A4B-it",
    max_seq_length = 2048,
    load_in_4bit = True,
    full_finetuning = False,
)
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16, lora_alpha = 16, lora_dropout = 0, bias = "none", random_state = 3407,
)

In [ ]:
# 5) Veriyi sohbet şablonuna çevir (Tulpar düşünme kapalı kullanıyor: "gemma-4")
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, train_on_responses_only
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-4")
veri = load_dataset("json", data_files = {"train": "egitim.jsonl", "test": "dogrulama.jsonl"})
def bicimle(ornekler):
    return {"text": [tokenizer.apply_chat_template(m, tokenize = False, add_generation_prompt = False).removeprefix("<bos>")
                     for m in ornekler["messages"]]}
veri = veri.map(bicimle, batched = True, remove_columns = ["messages"])
print(veri)
print(veri["train"][0]["text"][:800])

In [ ]:
# 6) Eğitim ayarları: sadece SENİN cevapların öğretilir (bağlam ezberlenmez)
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer,
    train_dataset = veri["train"], eval_dataset = veri["test"],
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        warmup_ratio = 0.03,
        lr_scheduler_type = "cosine",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        logging_steps = 20,
        eval_strategy = "steps", eval_steps = 200,
        save_strategy = "steps", save_steps = 200, save_total_limit = 2,
        output_dir = "kontrol_noktalari",
        bf16 = True, seed = 3407, report_to = "none",
    ),
)
trainer = train_on_responses_only(trainer, instruction_part = "<|turn>user\n", response_part = "<|turn>model\n")

In [ ]:
# 7) EĞİT (2-3 saat). Kesilirse bu hücreyi tekrar çalıştır: son kontrol noktasından devam eder.
import glob
devam = bool(glob.glob("kontrol_noktalari/checkpoint-*"))
istat = trainer.train(resume_from_checkpoint = devam)
print(istat)

In [ ]:
# 8) Hızlı deneme: doğrulama örneklerinde model ne yazıyor, sen ne yazmışsın?
import json
FastModel.for_inference(model)
ornekler = [json.loads(l) for l in open("dogrulama.jsonl")][:8]
for o in ornekler:
    girdi = tokenizer.apply_chat_template(o["messages"][:2], tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")
    cikti = model.generate(input_ids = girdi, max_new_tokens = 80, temperature = 0.7, top_p = 0.95)
    print("BAĞLAM (son satırlar):", "\n".join(o["messages"][1]["content"].splitlines()[-3:]))
    print("MODEL :", tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens = True).strip())
    print("SEN   :", o["messages"][2]["content"])
    print("-" * 60)

## 9) Kaydet ve indir
Önce **A** hücresini çalıştır (küçük LoRA dosyası, birkaç yüz MB). HP'de modelin üstüne takılır.
**B** yedektir: A HP'de çalışmazsa birleştirilmiş tam modeli üretir (uzun sürer, ~15 GB).

In [ ]:
# A) LoRA'yı kaydet, llama.cpp biçimine (GGUF) çevir, indir
model.save_pretrained("tulpar-lora"); tokenizer.save_pretrained("tulpar-lora")
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp > /dev/null
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_lora_to_gguf.py tulpar-lora --base-model-id unsloth/gemma-4-26B-A4B-it --outtype f16 --outfile tulpar-lora.gguf
!ls -lh tulpar-lora.gguf
files.download("tulpar-lora.gguf")

In [ ]:
# B) (Yedek) Birleştirilmiş model, HP'nin kullandığı boyutta (Q3_K_M)
# model.save_pretrained_gguf("tulpar-birlesik", tokenizer, quantization_method = "q3_k_m")
# !ls -lh tulpar-birlesik*